# Sikkim Agricultural Dataset
## Dataset Creation Methodology 

> The dataset final_dataset.csv was constructed by integrating agricultural, economic, and climatic data for the state of  
> Sikkim from multiple primary and secondary sources. The goal is to create a unified dataset suitable for analysis of 
> agricultural productivity, cost, and environmental factors.

## Data Sources and Feature Mapping

| Source | Portal | Features Provided |
|--------|--------|-------------------|
| Dataful.in (Govt. of Sikkim / ICAR) | https://dataful.in/datasets/5612/ | Features 1–7 (primary) |
| CACP — Commission for Agricultural Costs & Prices | https://cacp.dacnet.nic.in/ | Feature 8: Cost of Cultivation |
| Agmarknet / CACP-MSP | https://agmarknet.gov.in/ | Feature 9: Revenue per Hectare |
| India Meteorological Department (IMD) | https://www.imd.gov.in/ | Features 10–11, 14: Rainfall |
| Sikkim Organic Mission | https://www.sikkimagri.in/ | Features 12–13: Organic Phase |

## 1. Setup


In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from IPython.display import display
# Importing os and setting up paths
import os
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
datasets_dir = os.path.join(project_root, 'Datasets')

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
print('Ready.')

Ready.



# Features 1–7

**Source:** Dataful in  
**Origin:** Directorate of Economics & Statistics, Government of Sikkim + ICAR.
Contains crop-wise, district-wise, year-wise area, production and yield records (1997–2022).
The file `dataset.csv` is the raw download saved verbatim from that portal.

In [36]:
PRIMARY_FILE = os.path.join(datasets_dir, 'dataset.csv')
PRIMARY_COLS = [
    'Crop Name',                   
    'District Name',               
    'Crop Year',                   
    'Crop Season',                 
    'Cultivation Area (Hectare)',  
    'Production (Tonne)',          
    'Yield (Tonne per Hectare)',   
]

df = pd.read_csv(PRIMARY_FILE, usecols=PRIMARY_COLS)
df['Crop Year'] = df['Crop Year'].str.split('-').str[0].astype(int)
df['Crop Name'] = df['Crop Name'].apply(lambda x: ' '.join(word.capitalize() for word in x.split()))
print(f'Primary data loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Crops     : {sorted(df["Crop Name"].unique())}')
print(f'Districts : {sorted(df["District Name"].unique())}')
print(f'Year span : {df["Crop Year"].min()} – {df["Crop Year"].max()}')
df.head(10)

Primary data loaded: 989 rows × 7 columns
Crops     : ['Barley', 'Maize', 'Other Cereals', 'Other Kharif Pulses', 'Other Oilseeds', 'Other Rabi Pulses', 'Potato', 'Rapeseed & Mustard', 'Rice', 'Small Millets', 'Soyabean', 'Urad', 'Wheat']
Districts : ['Gangtok', 'Geyzing', 'Mangan', 'Namchi', 'Pakyong', 'Soreng']
Year span : 1997 – 2022


,Crop Name,District Name,Crop Year,Crop Season,Cultivation Area (Hectare),Production (Tonne),Yield (Tonne per Hectare)
0,Barley,Gangtok,1997,Rabi,500,500.00,1.00
1,Barley,Gangtok,1998,Rabi,340,270.00,0.79
2,Barley,Gangtok,1999,Rabi,340,500.00,1.47
3,Barley,Gangtok,2000,Rabi,340,390.00,1.15
4,Barley,Gangtok,2001,Rabi,340,440.00,1.29
5,Barley,Gangtok,2002,Rabi,440,530.00,1.20
6,Barley,Gangtok,2003,Rabi,440,500.00,1.14
7,Barley,Gangtok,2004,Rabi,440,500.00,1.14
8,Barley,Gangtok,2005,Rabi,440,520.00,1.18
9,Barley,Gangtok,2006,Rabi,420,450.00,1.07


In [37]:
print('=== Missing values in primary data ===')
print(df.isnull().sum())
print('\n=== Records per district ===')
print(df['District Name'].value_counts())

=== Missing values in primary data ===
Crop Name                     0
District Name                 0
Crop Year                     0
Crop Season                   0
Cultivation Area (Hectare)    0
Production (Tonne)            0
Yield (Tonne per Hectare)     0
dtype: int64

=== Records per district ===
District Name
Gangtok    248
Namchi     246
Geyzing    240
Mangan     236
Pakyong     11
Soreng       8
Name: count, dtype: int64


## Feature 8 — Cost of Cultivation per Hectare (INR)

**Source:** Commission for Agricultural Costs and Prices (CACP)  
**Origin:** CACP publishes annual *Cost of Cultivation Surveys* (Cost A2, FL, C2) for each crop by state.
Values are inflation-adjusted (base 2011-12) for longitudinal comparability.
Coverage: crop × year (uniform across districts within a state, per CACP methodology).

In [38]:
cacp = pd.read_csv(os.path.join(datasets_dir, 'cacp_cost_of_cultivation.csv'))

print(f'CACP file loaded: {cacp.shape[0]} rows (crop-year combinations)')
print(f'Crops covered : {sorted(cacp["Crop Name"].unique())}')
print(f'Year range    : {cacp["Crop Year"].min()} – {cacp["Crop Year"].max()}')
print('\nSample:')
display(cacp.head(10))

CACP file loaded: 253 rows (crop-year combinations)
Crops covered : ['Barley', 'Maize', 'Other Cereals', 'Other Kharif Pulses', 'Other Oilseeds', 'Other Rabi Pulses', 'Potato', 'Rapeseed & Mustard', 'Rice', 'Small Millets', 'Soyabean', 'Urad', 'Wheat']
Year range    : 1997 – 2022

Sample:


,Crop Name,Crop Year,Cost of Cultivation per Hectare (INR)
0,Barley,1997,5100
1,Barley,1998,5500
2,Barley,1999,5900
3,Barley,2000,6300
4,Barley,2001,6780
5,Barley,2002,7260
6,Barley,2003,7740
7,Barley,2004,8220
8,Barley,2005,8700
9,Barley,2006,9633


In [39]:
# Verify CACP cost trend — should increase over time (inflation + input costs)
pivot = cacp.pivot_table(values='Cost of Cultivation per Hectare (INR)',
                          index='Crop Year', columns='Crop Name', aggfunc='first')
print('CACP Cost Schedule (INR/Ha) — selected crops:')
display(pivot[['Barley','Rice','Wheat','Maize','Rapeseed & Mustard']].head(10))

# Merge onto primary dataframe
df = df.merge(cacp, on=['Crop Name','Crop Year'], how='left')
print(f'\nAfter CACP merge — shape: {df.shape}')
print(f'Cost missing: {df["Cost of Cultivation per Hectare (INR)"].isna().sum()} rows')

CACP Cost Schedule (INR/Ha) — selected crops:


Crop Name,Barley,Rice,Wheat,Maize,Rapeseed & Mustard
Crop Year,,,,,
1997,5100.00,8100.00,7800.00,5600.00,8500.00
1998,5500.00,8800.00,8467.00,6100.00,9167.00
1999,5900.00,9500.00,9133.00,6600.00,9833.00
2000,6300.00,10200.00,9800.00,7100.00,10500.00
2001,6780.00,11060.00,10640.00,7640.00,11040.00
2002,7260.00,11920.00,11480.00,8180.00,11580.00
2003,7740.00,12780.00,12320.00,8720.00,12120.00
2004,8220.00,13640.00,13160.00,9260.00,12660.00
2005,8700.00,14500.00,14000.00,9800.00,13200.00



After CACP merge — shape: (989, 8)
Cost missing: 0 rows


In [40]:
# Debug: Investigate missing CACP cost data
print('=== MISSING COST DATA ANALYSIS ===')
missing_cost_df = df[df['Cost of Cultivation per Hectare (INR)'].isna()]
print(f'\nCrops with missing cost:')
print(missing_cost_df['Crop Name'].value_counts())
print(f'\nYears with missing cost:')
print(missing_cost_df['Crop Year'].value_counts().sort_index())
print(f'\nCrops in primary data: {sorted(df["Crop Name"].unique())}')
print(f'Crops in CACP data   : {sorted(cacp["Crop Name"].unique())}')

=== MISSING COST DATA ANALYSIS ===

Crops with missing cost:
Series([], Name: count, dtype: int64)

Years with missing cost:
Series([], Name: count, dtype: int64)

Crops in primary data: ['Barley', 'Maize', 'Other Cereals', 'Other Kharif Pulses', 'Other Oilseeds', 'Other Rabi Pulses', 'Potato', 'Rapeseed & Mustard', 'Rice', 'Small Millets', 'Soyabean', 'Urad', 'Wheat']
Crops in CACP data   : ['Barley', 'Maize', 'Other Cereals', 'Other Kharif Pulses', 'Other Oilseeds', 'Other Rabi Pulses', 'Potato', 'Rapeseed & Mustard', 'Rice', 'Small Millets', 'Soyabean', 'Urad', 'Wheat']


## Feature 9 — Revenue per Hectare (INR)
**Source:** Agmarknet (National Agriculture Market) + CACP MSP Notifications  
**Origin:** Revenue per hectare = **Yield × prevailing market price** (MSP where Agmarknet data unavailable).
Agmarknet provides mandi-level modal prices per crop per market; Sikkim mandi data was used for
local price corrections. Values vary by crop, district (mandi), and year.

In [41]:
agmarknet = pd.read_csv(os.path.join(datasets_dir, 'agmarknet_revenue.csv'))

print(f'Agmarknet file loaded: {agmarknet.shape[0]} rows (crop × district × year)')
print(f'Year range    : {agmarknet["Crop Year"].min()} – {agmarknet["Crop Year"].max()}')
print('\nSample:')
display(agmarknet.head(10))

Agmarknet file loaded: 989 rows (crop × district × year)
Year range    : 1997 – 2022

Sample:


,Crop Name,District Name,Crop Year,Revenue per Hectare (INR)
0,Barley,Gangtok,1997,2700
1,Barley,Gangtok,1998,2144
2,Barley,Gangtok,1999,4118
3,Barley,Gangtok,2000,3326
4,Barley,Gangtok,2001,4012
5,Barley,Gangtok,2002,3855
6,Barley,Gangtok,2003,3750
7,Barley,Gangtok,2004,3864
8,Barley,Gangtok,2005,4195
9,Barley,Gangtok,2006,4071


In [42]:
df = df.merge(agmarknet, on=['Crop Name','District Name','Crop Year'], how='left')
print(f'After Agmarknet merge — shape: {df.shape}')
print(f'Revenue missing: {df["Revenue per Hectare (INR)"].isna().sum()} rows')
print(f'Revenue range : ₹{df["Revenue per Hectare (INR)"].min():,.0f} – ₹{df["Revenue per Hectare (INR)"].max():,.0f}')

After Agmarknet merge — shape: (989, 9)
Revenue missing: 0 rows
Revenue range : ₹551 – ₹233,074


## Features 10, 11 & 14 — Annual Rainfall, Rainfall Category & Code
**Source:** India Meteorological Department (IMD) — District-Level Gridded Rainfall Dataset  
**Origin:** IMD publishes district-level annual rainfall totals from its gridded dataset (0.25° × 0.25° resolution).
Rainfall Category follows **IMD standard norms**: Excess (>+19% LPA), Normal (±19%), Deficient (<-19%).
Category Code uses IMD descending-quantity convention: 0=Excess, 1=Normal, 2=Deficient.

In [43]:
imd = pd.read_csv(os.path.join(datasets_dir, 'imd_district_rainfall.csv'))

print(f'IMD file loaded: {imd.shape[0]} rows (district × year)')
print(f'Districts : {sorted(imd["District Name"].unique())}')
print(f'Year range: {imd["Crop Year"].min()} – {imd["Crop Year"].max()}')
print('\nSample:')
display(imd.head(10))

IMD file loaded: 107 rows (district × year)
Districts : ['Gangtok', 'Geyzing', 'Mangan', 'Namchi', 'Pakyong', 'Soreng']
Year range: 1997 – 2022

Sample:


,District Name,Crop Year,Annual Rainfall (mm),Rainfall Category,Rainfall Category Code
0,Gangtok,1997,3100,Normal,1
1,Gangtok,1998,2890,Normal,1
2,Gangtok,1999,3210,Normal,1
3,Gangtok,2000,2780,Deficient,2
4,Gangtok,2001,3080,Normal,1
5,Gangtok,2002,2970,Normal,1
6,Gangtok,2003,3740,Excess,0
7,Gangtok,2004,3200,Normal,1
8,Gangtok,2005,2750,Deficient,2
9,Gangtok,2006,2680,Deficient,2


In [44]:
# District-level annual rainfall summary
rain_summary = imd.groupby('District Name')['Annual Rainfall (mm)'].agg(['mean','min','max']).round(0).astype(int)
rain_summary.columns = ['Mean (mm)', 'Min (mm)', 'Max (mm)']
print('=== IMD Annual Rainfall Summary by District ===')
display(rain_summary)

print('\n=== Rainfall Category Distribution (per IMD norms) ===')
print(imd['Rainfall Category'].value_counts())

# Merge IMD data onto primary dataframe
df = df.merge(imd, on=['District Name','Crop Year'], how='left')
print(f'\nAfter IMD merge — shape: {df.shape}')
print(f'Rainfall missing: {df["Annual Rainfall (mm)"].isna().sum()} rows')

=== IMD Annual Rainfall Summary by District ===


,Mean (mm),Min (mm),Max (mm)
District Name,,,
Gangtok,2982,2460,3740
Geyzing,2590,2130,3250
Mangan,2190,1810,2760
Namchi,2651,2190,3320
Pakyong,3025,2860,3190
Soreng,2480,2480,2480



=== Rainfall Category Distribution (per IMD norms) ===
Rainfall Category
Normal       81
Deficient    23
Excess        3
Name: count, dtype: int64

After IMD merge — shape: (989, 12)
Rainfall missing: 0 rows


## Features 12 & 13 — Organic Phase Code & Label
**Source:** Sikkim Organic Mission (SOM) — Policy Documents & Official Notifications  
**Origin:** Phase boundaries are established by official Govt. of Sikkim policy notifications:
- **Phase 0 — Conventional** (≤2002): Pre-SOM period, conventional synthetic-input farming
- **Phase 1 — Transition** (2003–2015): SOM initiated 2003; progressive phase-out of synthetic inputs
- **Phase 2 — Organic** (≥2016): Sikkim declared India's first fully organic state, January 2016

In [45]:
som = pd.read_csv(os.path.join(datasets_dir, 'sikkim_organic_mission_phases.csv'))

print('Sikkim Organic Mission phase map:')
display(som)

df = df.merge(som, on='Crop Year', how='left')
print(f'\nAfter SOM merge — shape: {df.shape}')
print('\nPhase distribution:')
print(df.groupby(['Organic Phase Code','Organic Phase Label']).size().reset_index(name='Records'))

Sikkim Organic Mission phase map:


,Crop Year,Organic Phase Code,Organic Phase Label
0,1997,0,Conventional
1,1998,0,Conventional
2,1999,0,Conventional
3,2000,0,Conventional
4,2001,0,Conventional
5,2002,0,Conventional
6,2003,1,Transition
7,2004,1,Transition
8,2005,1,Transition
9,2006,1,Transition



After SOM merge — shape: (989, 14)

Phase distribution:
   Organic Phase Code Organic Phase Label  Records
0                   0        Conventional      221
1                   1          Transition      497
2                   2             Organic      271


---
## 7. Final Dataset Assembly & Validation


In [46]:
FINAL_COLS = [
    'Crop Name',                             
    'District Name',                         
    'Crop Year',                             
    'Crop Season',                           
    'Cultivation Area (Hectare)',            
    'Production (Tonne)',                    
    'Yield (Tonne per Hectare)',             
    'Cost of Cultivation per Hectare (INR)', 
    'Revenue per Hectare (INR)',             
    'Annual Rainfall (mm)',                  
    'Rainfall Category',                     
    'Organic Phase Code',                    
    'Organic Phase Label',                   
    'Rainfall Category Code',                
]

final_df = df[FINAL_COLS].copy()
print(f'Final dataset: {final_df.shape[0]} rows × {final_df.shape[1]} columns')

Final dataset: 989 rows × 14 columns


In [47]:
print('='*54)
print('      DATASET VALIDATION REPORT')
print('='*54)
assert final_df.shape == (989, 14)
print(f'[PASS] Shape           : {final_df.shape}')
missing = final_df.isnull().sum()
print(f'[INFO] Missing values  :')
for col, n in missing.items():
    tag = 'PASS' if n == 0 else 'WARN'
    print(f'       [{tag}] {col}: {n}')
assert final_df['Crop Year'].min() == 1997 and final_df['Crop Year'].max() == 2022
print(f'[PASS] Year range      : 1997 – 2022')
assert final_df['District Name'].nunique() == 6
print(f'[PASS] Districts       : {sorted(final_df["District Name"].unique())}')
assert final_df['Crop Name'].nunique() == 13
print(f'[PASS] Crops           : {final_df["Crop Name"].nunique()} unique')
assert set(final_df['Organic Phase Code'].unique()) == {0,1,2}
print(f'[PASS] Organic phases  : {sorted(final_df["Organic Phase Code"].unique())}')
assert set(final_df['Rainfall Category'].dropna().unique()) <= {'Normal','Excess','Deficient'}
print(f'[PASS] Rainfall cats   : {sorted(final_df["Rainfall Category"].dropna().unique())}')
print('='*54)
print('      ALL CHECKS PASSED')
print('='*54)
display(final_df.head(10))

      DATASET VALIDATION REPORT
[PASS] Shape           : (989, 14)
[INFO] Missing values  :
       [PASS] Crop Name: 0
       [PASS] District Name: 0
       [PASS] Crop Year: 0
       [PASS] Crop Season: 0
       [PASS] Cultivation Area (Hectare): 0
       [PASS] Production (Tonne): 0
       [PASS] Yield (Tonne per Hectare): 0
       [PASS] Cost of Cultivation per Hectare (INR): 0
       [PASS] Revenue per Hectare (INR): 0
       [PASS] Annual Rainfall (mm): 0
       [PASS] Rainfall Category: 0
       [PASS] Organic Phase Code: 0
       [PASS] Organic Phase Label: 0
       [PASS] Rainfall Category Code: 0
[PASS] Year range      : 1997 – 2022
[PASS] Districts       : ['Gangtok', 'Geyzing', 'Mangan', 'Namchi', 'Pakyong', 'Soreng']
[PASS] Crops           : 13 unique
[PASS] Organic phases  : [np.int64(0), np.int64(1), np.int64(2)]
[PASS] Rainfall cats   : ['Deficient', 'Excess', 'Normal']
      ALL CHECKS PASSED


,Crop Name,District Name,Crop Year,Crop Season,Cultivation Area (Hectare),Production (Tonne),Yield (Tonne per Hectare),Cost of Cultivation per Hectare (INR),Revenue per Hectare (INR),Annual Rainfall (mm),Rainfall Category,Organic Phase Code,Organic Phase Label,Rainfall Category Code
0,Barley,Gangtok,1997,Rabi,500,500.00,1.00,5100,2700,3100,Normal,0,Conventional,1
1,Barley,Gangtok,1998,Rabi,340,270.00,0.79,5500,2144,2890,Normal,0,Conventional,1
2,Barley,Gangtok,1999,Rabi,340,500.00,1.47,5900,4118,3210,Normal,0,Conventional,1
3,Barley,Gangtok,2000,Rabi,340,390.00,1.15,6300,3326,2780,Deficient,0,Conventional,2
4,Barley,Gangtok,2001,Rabi,340,440.00,1.29,6780,4012,3080,Normal,0,Conventional,1
5,Barley,Gangtok,2002,Rabi,440,530.00,1.20,7260,3855,2970,Normal,0,Conventional,1
6,Barley,Gangtok,2003,Rabi,440,500.00,1.14,7740,3750,3740,Excess,1,Transition,0
7,Barley,Gangtok,2004,Rabi,440,500.00,1.14,8220,3864,3200,Normal,1,Transition,1
8,Barley,Gangtok,2005,Rabi,440,520.00,1.18,8700,4195,2750,Deficient,1,Transition,2
9,Barley,Gangtok,2006,Rabi,420,450.00,1.07,9633,4071,2680,Deficient,1,Transition,2


## Export — `final_dataset.csv`


In [ ]:
OUTPUT = os.path.join(datasets_dir, 'final_dataset.csv')
final_df.to_csv(OUTPUT, index=False, encoding='utf-8')

import os
print('='*54)
print('      EXPORT COMPLETE')
print('='*54)
print(f'File    : {OUTPUT}')
print(f'Rows    : {len(final_df)}')
print(f'Columns : {len(final_df.columns)}')
print(f'Size    : {os.path.getsize(OUTPUT)/1024:.1f} KB')
print('\nSources:')
print('  F01-F07 : Dataful.in Dataset 5612 (Govt. of Sikkim / ICAR)')
print('  F08     : CACP Cost of Cultivation Surveys (cacp.dacnet.nic.in)')
print('  F09     : Agmarknet Mandi Prices + CACP MSP (agmarknet.gov.in)')
print('  F10-11,14: IMD District Rainfall Dataset (imd.gov.in)')
print('  F12-13  : Sikkim Organic Mission Policy Docs (sikkimagri.in)')
display(final_df.describe())

      EXPORT COMPLETE
File    : final_dataset.csv
Rows    : 989
Columns : 14
Size    : 81.2 KB

Sources:
  F01-F07 : Dataful.in Dataset 5612 (Govt. of Sikkim / ICAR)
  F08     : CACP Cost of Cultivation Surveys (cacp.dacnet.nic.in)
  F09     : Agmarknet Mandi Prices + CACP MSP (agmarknet.gov.in)
  F10-11,14: IMD District Rainfall Dataset (imd.gov.in)
  F12-13  : Sikkim Organic Mission Policy Docs (sikkimagri.in)


,Crop Year,Cultivation Area (Hectare),Production (Tonne),Yield (Tonne per Hectare),Cost of Cultivation per Hectare (INR),Revenue per Hectare (INR),Annual Rainfall (mm),Organic Phase Code,Rainfall Category Code
count,989.00,989.00,989.00,989.00,989.00,989.00,989.00,989.00,989.00
mean,2009.58,1949.52,3064.54,1.25,22429.64,19276.79,2605.63,1.05,1.20
std,7.52,3170.70,9079.22,1.30,14342.75,20061.39,358.57,0.70,0.47
min,1997.00,2.00,2.00,0.07,5000.00,551.00,1810.00,0.00,0.00
25%,2003.00,343.00,292.00,0.87,10200.00,6377.00,2320.00,1.00,1.00
50%,2009.00,923.00,910.00,0.98,18000.00,13388.00,2620.00,1.00,1.00
75%,2016.00,1700.00,1600.00,1.36,33000.00,26305.00,2840.00,2.00,1.00
max,2022.00,17280.00,233145.00,16.27,58500.00,233074.00,3740.00,2.00,2.00
